In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
from datasets import load_dataset
from scipy.stats import pearsonr, spearmanr
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# Load Dataset
dataset = load_dataset("mteb/stsbenchmark-sts")

# Rename 'score' to 'label' so the Trainer detects it automatically
dataset = dataset.rename_column("score", "label")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl.gz:   0%|          | 0.00/278k [00:00<?, ?B/s]

validation.jsonl.gz:   0%|          | 0.00/86.4k [00:00<?, ?B/s]

test.jsonl.gz:   0%|          | 0.00/63.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.squeeze()
    return {
        "pearson": pearsonr(predictions, labels)[0],
        "spearman": spearmanr(predictions, labels)[0]
    }

In [ ]:
import numpy as np
from datasets import load_dataset
from scipy.stats import pearsonr, spearmanr
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
def cast_label(examples):
    examples["label"] = [float(x) for x in examples["label"]]
    return examples
dataset = dataset.map(cast_label, batched=True)
models_to_train = ["roberta-base", "google/electra-base-discriminator"]

for model_name in models_to_train:
    print(f"\n{'='*20} Training {model_name} {'='*20}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_fn(examples):
        return tokenizer(
            examples["sentence1"],
            examples["sentence2"],
            truncation=True,
            padding=False,
        )

    tokenized_ds = dataset.map(tokenize_fn, batched=True)

    # remove unused text columns
    cols_to_remove = [c for c in ["sentence1", "sentence2"] if c in tokenized_ds["train"].column_names]
    tokenized_ds = tokenized_ds.remove_columns(cols_to_remove)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)

    args = TrainingArguments(
        output_dir=f"/content/drive/MyDrive/RamsayData/Outcome/{model_name.replace('/', '_')}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        load_best_model_at_end=True,
        report_to="none",
        logging_strategy="epoch",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["validation"],
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )

    trainer.train()
    results = trainer.evaluate(tokenized_ds["test"])

    print(f"\n>>> TEST RESULTS for {model_name}:")
    print(f"Pearson:  {results['eval_pearson']:.4f}")
    print(f"Spearman: {results['eval_spearman']:.4f}")

Map:   0%|          | 0/5749 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1379 [00:00<?, ? examples/s]


==================== Training roberta-base ====================


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye


>>> TEST RESULTS for google/electra-base-discriminator:
Pearson:  0.8963
Spearman: 0.8929


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
import os

# GPU ISSUES CODE DON'T MIND

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

device = "cpu"

MODEL_DIR = "/content/drive/MyDrive/RamsayData/Outcome/google_electra-base-discriminator/checkpoint-1080"

print(f"Listing contents of {MODEL_DIR}:")
if os.path.exists(MODEL_DIR):
    for item in os.listdir(MODEL_DIR):
        print(f"- {item}")
else:
    print(f"Directory does not exist: {MODEL_DIR}")
print("-" * 60)

try:
    config = AutoConfig.from_pretrained(MODEL_DIR)
    print(f"Config num_labels: {config.num_labels}")
except Exception as e:
    print(f"Error loading config: {e}")
print("-" * 60)


tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, fix_mistral_regex=True)

try:
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device) # .to(device) will now send it to CPU
    print("Model loaded successfully on CPU.")

    print("Running on CPU to bypass persistent GPU error.")
except Exception as e:
    print(f"Error loading or moving model to device: {e}")
    model = None


# HERE STARTS THE MODEL EVALUATION
if model:
    model.eval()

    def predict_similarity(s1, s2, clip_0_5=True):
        inputs = tokenizer(s1, s2, return_tensors="pt", truncation=True, padding=True).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        score = outputs.logits.squeeze().item()
        if clip_0_5:
            score = max(0.0, min(5.0, score))
        return score

    tests = [
        ("A man is playing guitar.", "A man is playing music."),
        ("A man is playing guitar.", "The stock market closed higher today."),
    ]

    for a, b in tests:
        print("S1:", a)
        print("S2:", b)
        print("Predicted similarity:", predict_similarity(a, b))
        print("-" * 60)
else:
    print("Model could not be loaded, skipping predictions.")

Listing contents of /content/drive/MyDrive/RamsayData/Outcome/google_electra-base-discriminator/checkpoint-1080:
- tokenizer_config.json
- config.json
- rng_state.pth
- model.safetensors
- scheduler.pt
- trainer_state.json
- training_args.bin
- optimizer.pt
- tokenizer.json
------------------------------------------------------------
Config num_labels: 1
------------------------------------------------------------


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully on CPU.
Running on CPU to bypass persistent GPU error.
S1: A man is playing guitar.
S2: A man is playing music.
Predicted similarity: 3.19637131690979
------------------------------------------------------------
S1: A man is playing guitar.
S2: The stock market closed higher today.
Predicted similarity: 0.15913276374340057
------------------------------------------------------------
